In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


# Re-ranking

O Amazon Bedrock fornece acesso a modelos de reranker que você pode usar ao consultar para melhorar a relevância dos resultados recuperados. O modelo de reranker calcula a relevância dos chunks em relação a uma query e reordena os resultados com base nos scores calculados. Ao usar um modelo de reranker, você pode retornar respostas mais adequadas para responder à query.

Modelos de reranker são treinados para identificar sinais de relevância com base em uma query e então usar esses sinais para classificar documentos. Por isso, os modelos podem fornecer resultados mais relevantes e precisos.

Se você está usando `Amazon Bedrock Knowledge Bases` para construir sua aplicação de Retrieval Augmented Generation (RAG), use um modelo de reranker ao chamar a operação `Retrieve` ou `RetrieveAndGenerate`. Os resultados do reranking substituem o ranking padrão que o Amazon Bedrock Knowledge Bases determina.

Este notebook demonstra o uso do **modelo de reranking** com Amazon Bedrock Knowledge Bases, por meio da API Rerank, que ajudará a melhorar ainda mais a precisão e a relevância das aplicações RAG. Com um modelo de reranker, você pode recuperar menos resultados, porém mais relevantes. Ao fornecer esses resultados ao foundation model que você usa para gerar uma resposta, você também pode reduzir custo e latência.

Vamos explorar como implementar e utilizar modelos de reranking com Amazon Bedrock Knowledge Bases para um caso de uso de exemplo.

## 1. Setup
Antes de executar o restante deste notebook, você precisará executar as células abaixo para (garantir que as bibliotecas necessárias estejam instaladas e) conectar-se ao Bedrock.

Por favor, ignore qualquer erro de dependência do pip (se você ver algum durante a instalação das bibliotecas)

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
%pip install --upgrade boto3

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
import boto3
print(boto3.__version__)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

Este código faz parte do setup e é usado para:
- Adicionar o diretório pai ao path do sistema Python
- Importar um módulo customizado (BedrockStructuredKnowledgeBase) do `utils` necessário para execuções posteriores

In [ ]:
import os
import sys
import time
import boto3
import logging
import pprint
import json

# Set the path to import module
from pathlib import Path
current_path = Path().resolve()
current_path = current_path.parent
if str(current_path) not in sys.path:
    sys.path.append(str(current_path))
# Print sys.path to verify
# print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session(region_name = 'us-west-2')
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import time

# Get the current timestamp
current_time = time.time()

# Format the timestamp as a string
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime(current_time))[-7:]
# Create the suffix using the timestamp
suffix = f"{timestamp_str}"
knowledge_base_name = 'reranking-kb'
knowledge_base_description = "Knowledge Base for re-ranking."
bucket_name = f'{knowledge_base_name}-{suffix}'
DEFAULT_BEDROCK_MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", DEFAULT_BEDROCK_MODEL_ID)

def bedrock_model_arn(model_id):
    return _bedrock_model_arn(model_id, region)

# Define data sources
data_source=[{"type": "S3", "bucket_name": bucket_name}]

## 2 - Criar knowledge bases com estratégia de fixed chunking
Vamos começar criando uma [Amazon Bedrock Knowledge Bases](https://aws.amazon.com/bedrock/knowledge-bases/) para armazenar dados de videogames em formato csv. Knowledge Bases permitem integrar com diferentes bancos de dados vetoriais incluindo [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/), [Amazon Aurora](https://aws.amazon.com/rds/aurora/), [Pinecone](http://app.pinecone.io/bedrock-integration), [Redis Enterprise]() e [MongoDB Atlas](). Para este exemplo, integraremos a knowledge base com Amazon OpenSearch Serverless. Para isso, usaremos a classe auxiliar `BedrockKnowledgeBase` que criará a knowledge base e todos os seus pré-requisitos:
1. Roles e policies IAM
2. Bucket S3
3. Políticas de encryption, network e data access do Amazon OpenSearch Serverless
4. Coleção Amazon OpenSearch Serverless
5. Índice vetorial do Amazon OpenSearch Serverless
6. Knowledge base
7. Data source da Knowledge base

Criaremos uma knowledge base usando a estratégia de fixed chunking.

Você pode escolher diferentes estratégias de chunking alterando os valores do parâmetro abaixo:
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_metadata = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source, 
    chunking_strategy = "FIXED_SIZE", 
    suffix = suffix
)

### 2.1 Baixar os relatórios anuais da Amazon de 2019, 2020, 2021, 2022 e 2023 e fazer upload para o Amazon S3

Agora que criamos a knowledge base, vamos populá-la com o dataset de relatórios `sec-10-k` na KB. Esses dados estão sendo baixados [daqui](https://ir.aboutamazon.com/annual-reports-proxies-and-shareholder-letters/default.aspx). Esses dados são sobre os relatórios anuais, proxies e cartas aos acionistas da Amazon.

In [ ]:
import os

def create_directory(directory_name):    
    if not os.path.exists(directory_name):
        os.makedirs(directory_name)
        print(f"Directory '{directory_name}' created successfully.")
    else:
        print(f"Directory '{directory_name}' already exists.")

# Call the function to create the directory
create_directory("sec-10-k")

In [ ]:
import requests

def download_file(url, filename):
    # Send a GET request to the URL
    response = requests.get(url)
    
    # Check if the request was successful
    if response.status_code == 200:
        # Open the file in write-binary mode
        with open(filename, 'wb') as file:
            # Write the content of the response to the file
            file.write(response.content)
        print(f"File downloaded successfully: {filename}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

# URL of the files to download
urls = ["https://s2.q4cdn.com/299287126/files/doc_financials/2024/ar/Amazon-com-Inc-2023-Annual-Report.pdf",
        "https://s2.q4cdn.com/299287126/files/doc_financials/2023/ar/Amazon-2022-Annual-Report.pdf",
        "https://s2.q4cdn.com/299287126/files/doc_financials/2022/ar/Amazon-2021-Annual-Report.pdf",
        "https://s2.q4cdn.com/299287126/files/doc_financials/2021/ar/Amazon-2020-Annual-Report.pdf",
        "https://s2.q4cdn.com/299287126/files/doc_financials/2020/ar/2019-Annual-Report.pdf"]


for url in urls:
    # Name for the downloaded file
    filename = url.split('/')[-1]

    # Path to save the downloaded file
    filepath = f"./sec-10-k/{filename}"

    # Call the function to download the file
    download_file(url, filepath)

Vamos fazer upload dos dados de relatórios anuais disponíveis na pasta `sec-10-k` para o S3.

In [ ]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                if not file.startswith('.DS_Store'):
                    file_to_upload = os.path.join(root,file)
                    print(f"uploading file {file_to_upload} to {bucket_name}")
                    s3_client.upload_file(file_to_upload,bucket_name,file)

# upload metadata file to S3
upload_directory("sec-10-k", bucket_name)

Agora inicie o ingestion job. Como estamos usando os mesmos documentos utilizados para fixed chunking, estamos pulando a etapa de upload dos documentos para o bucket S3.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_metadata.start_ingestion_job()

Por fim, salvamos o Knowledge Base Id para testar a solução em uma etapa posterior.

In [ ]:
kb_id = knowledge_base_metadata.get_knowledge_base_id()

## 3. Avaliar a relevância das respostas das queries com e sem Re-ranking (usando Ragas)

Definir modelos para geração, avaliação e re-ranking

In [ ]:
from langchain.llms.bedrock import Bedrock
from langchain_aws import ChatBedrock
from langchain_aws import BedrockEmbeddings

bedrock_client = boto3.client('bedrock-runtime')

TEXT_GENERATION_MODEL_ID = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")
EVALUATION_MODEL_ID = os.getenv("BEDROCK_EVALUATION_MODEL_ID", "anthropic.claude-sonnet-4-6")
EMBEDDING_MODEL_ID = os.getenv("BEDROCK_EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")

# Reranker model: there are two reranker models available at launch
AMAZON_RERANKER_MODEL_ID = "amazon.rerank-v1:0"
COHERE_RERANKER_MODEL_ID = "cohere.rerank-v3-5:0"


llm_for_evaluation = ChatBedrock(model_id=EVALUATION_MODEL_ID, client=bedrock_client)
bedrock_embeddings = BedrockEmbeddings(model_id=EMBEDDING_MODEL_ID, client=bedrock_client)


#### 3.1 Atualizar a execution role da Knowledge Base

In [ ]:
# Before using autogenerated filters - update the knowledge base execution IAM role with right permissions

iam = boto3.resource('iam')
client = boto3.client('iam')

def get_attached_policies(role_name):
    response = client.list_attached_role_policies(RoleName=role_name)
    attached_policies = response['AttachedPolicies']
    return attached_policies

# get the knowledge base IAM role name
get_kb_response = bedrock_agent_client.get_knowledge_base(knowledgeBaseId = kb_id)
role_arn = get_kb_response['knowledgeBase']['roleArn']
role_name = role_arn.split('/')[-1]

# get attached policies
attached_policies = get_attached_policies(role_name)
attached_policies

def update_kb_execution_role(attached_policies, region_name):
    
    for policy in attached_policies:

        print(policy['PolicyArn'])
        policy_name = policy['PolicyName']
        policy_arn = policy['PolicyArn']

        if 'FoundationModel' in policy_arn:
            print('Updating FoundationModel policy: ',policy_arn)
            policy = iam.Policy(policy_arn)
            version = policy.default_version
            policyJson = version.document
            policyJson['Statement'][0]['Resource'].append(bedrock_model_arn(TEXT_GENERATION_MODEL_ID))
            policyJson['Statement'][0]['Resource'].append(bedrock_model_arn(EVALUATION_MODEL_ID))  
            policyJson['Statement'][0]['Resource'].append(bedrock_model_arn(AMAZON_RERANKER_MODEL_ID)) 
            policyJson['Statement'][0]['Resource'].append(bedrock_model_arn(COHERE_RERANKER_MODEL_ID)) 
        
            client.detach_role_policy(RoleName=role_name,
                PolicyArn=policy_arn)
            
            response = client.delete_policy(
                PolicyArn=policy_arn
            )
            print(response)
           
            response = client.create_policy(
            PolicyName= policy_name,
            PolicyDocument=json.dumps(policyJson)
            )
            print(response)
        
        client.attach_role_policy(
            RoleName=role_name,
            PolicyArn=policy_arn
        )

In [ ]:
update_kb_execution_role(attached_policies, region)
# time.sleep(30)

#### 3.2 Personalizar a configuração de retrieve and generate

In [ ]:
def retrieve_and_generate(query, reranker_model=None, kb_id=None, TEXT_GENERATION_MODEL_ID=None, metadata_filters=None):
    
    # Prepare retrieval configuration
    retrieval_config = {
        "vectorSearchConfiguration": {
            "numberOfResults": 30 if reranker_model else 3
        }
    }

    if reranker_model:
        retrieval_config["vectorSearchConfiguration"]["rerankingConfiguration"] = {
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "modelConfiguration": {
                    "modelArn": bedrock_model_arn(reranker_model),
                },
                "numberOfRerankedResults": 3
            }
        }

        if metadata_filters:
            retrieval_config["vectorSearchConfiguration"]["rerankingConfiguration"]["bedrockRerankingConfiguration"]["metadataConfiguration"] = {
                                                                "selectionMode" : "SELECTIVE",
                                                                "selectiveModeConfiguration" : {
                                                                    "fieldsToInclude": [{
                                                                        "fieldName": "year",
                                                                    }]
                                                                }
                                                            }
                    

    # Call the retrieve and generate API
    start = time.time()
    response = bedrock_agent_runtime_client.retrieve_and_generate(
        input={'text': query},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': bedrock_model_arn(TEXT_GENERATION_MODEL_ID),
                'retrievalConfiguration': retrieval_config,
            },
        }
    )
    time_spent = time.time() - start

    print(f"[Response] : {response['output']['text']}\n")
    print(f"[Invocation time] : {time_spent}\n")

    return response


#### 3.3 Preparar dataset para avaliação

In [ ]:
import json
import re
import pandas as pd
from datasets import Dataset, Features, Sequence, Value
from ragas import evaluate
from ragas.metrics import (
    answer_correctness
)

#specify the metrics here
metrics = [
    answer_correctness
]

def bounded_evaluation(dataset):
    rows = []
    for sample in dataset:
        prompt = (
            "Evaluate the answer against the question and ground truth. "
            "Return only JSON with answer_correctness from 0 to 1.\n"
            f"Question: {sample['question']}\n"
            f"Ground truth: {sample['ground_truth']}\n"
            f"Answer: {sample['answer']}\n"
            f"Context: {' '.join(sample['contexts'][:5])}"
        )
        payload = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 64,
            "temperature": 0,
            "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
        })
        response = bedrock_client.invoke_model(
            body=payload,
            modelId=EVALUATION_MODEL_ID,
            accept="application/json",
            contentType="application/json",
        )
        text = json.loads(response["body"].read())["content"][0]["text"]
        match = re.search(r"\{.*\}", text, re.DOTALL)
        scores = json.loads(match.group(0)) if match else {}
        rows.append({
            "question": sample["question"],
            "answer": sample["answer"],
            "ground_truth": sample["ground_truth"],
            "contexts": sample["contexts"],
            "answer_correctness": float(scores.get("answer_correctness", 0)),
            "evaluation_mode": "bounded_bedrock_smoke",
        })
    return pd.DataFrame(rows)

questions = [
    "How many jobs did Amazon create in 2020, and what was its total global workforce after this expansion?",
    "How does the 2023 net sales mix reflect Amazon's global priorities and strategic investments across segments?"
]
ground_truths = [
    "Amazon added 500,000 jobs in 2020, bringing its total workforce to approximately 1.3 million employees worldwide.",
    "Amazon's 2023 net sales mix highlights its global priorities, with North America contributing 61%, International 23%, and AWS 16% of total sales. Year-over-year growth in each segment—12% for North America, 11% for International, and 13% for AWS—was driven by increased unit sales, advertising services, and subscription offerings. These trends reflect Amazon's balanced approach to expanding its core markets, strengthening its international presence, and investing in AWS's innovative cloud services to sustain long-term growth."
    ]

In [ ]:
def prepare_eval_dataset(questions, ground_truths, kb_id=None, TEXT_GENERATION_MODEL_ID=None, reranker_model=None, metadata_filters = None):
    answers = []
    contexts = []
    
    for query in questions:
        response = retrieve_and_generate(
            query,
            reranker_model=reranker_model,
            kb_id=kb_id,
            TEXT_GENERATION_MODEL_ID=TEXT_GENERATION_MODEL_ID,
            metadata_filters=metadata_filters
        )
        
        answers.append(response["output"]["text"])
        
        context_group = []
        for citation in response.get("citations", []):
            context_group.extend([
                ref["content"]["text"]
                for ref in citation.get("retrievedReferences", [])
                if "content" in ref and "text" in ref["content"]
            ])
        contexts.append(context_group)
        time.sleep(15)

    # Create dictionary
    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    }

    # Convert dict to dataset
    dataset = Dataset.from_dict(
        data,
        features=Features({
            "question": Value("string"),
            "answer": Value("string"),
            "contexts": Sequence(Value("string")),
            "ground_truth": Value("string"),
        }),
    )
    return dataset


#### 3.4 Avaliar dataset - sem re-ranker

In [ ]:
without_reranker_dataset = prepare_eval_dataset(questions, ground_truths, kb_id, TEXT_GENERATION_MODEL_ID, reranker_model=None)

In [ ]:
without_reranker_result_df = bounded_evaluation(without_reranker_dataset)

#### 3.5 Avaliar dataset - com re-ranker

In [ ]:
with_reranker_dataset = prepare_eval_dataset(questions, ground_truths, kb_id, TEXT_GENERATION_MODEL_ID, reranker_model=AMAZON_RERANKER_MODEL_ID)

In [ ]:
with_reranker_result_df = bounded_evaluation(with_reranker_dataset)

#### 3.4 Avaliar dataset - com re-ranker + configuração de metadata

##### 3.4.1 Preparar metadata para ingestão

In [ ]:
import json
import re

def generate_matadata(data_dir):
    
    # Loop through all PDF files in the directory
    for filename in os.listdir(data_dir):
        if not filename.startswith('.DS_Store'):
            # Define the metadata dictionary
            metadata ={}
            
            filename= f'{data_dir}/{filename}'
            print(filename)
            
            # Create metadata
            metadata["company"] = "Amazon"
            metadata["ticker"] = "AMZN"
            metadata["year"] = re.search(r'\d+', filename.split('/')[-1]).group(0)

            # Create a JSON object
            json_data = {"metadataAttributes": metadata}

            # print(json_data)

            # Write the JSON object to a file
            with open(f"{filename.replace('.pdf', '.pdf.metadata.json')}", "w") as f:
                json.dump(json_data, f)


In [ ]:
data_dir = './sec-10-k'
generate_matadata(data_dir)

In [ ]:
# upload metadata file to S3
upload_directory("sec-10-k", bucket_name)

##### 3.4.2 Ingerir metadata nas Knowledge Bases

Agora inicie o ingestion job. Como estamos usando os mesmos documentos utilizados para fixed chunking, estamos pulando a etapa de upload dos documentos para o bucket S3.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_metadata.start_ingestion_job()

In [ ]:
with_reranker_metadata_filters_dataset = prepare_eval_dataset(questions, ground_truths, kb_id, TEXT_GENERATION_MODEL_ID, reranker_model=AMAZON_RERANKER_MODEL_ID, metadata_filters=True)

In [ ]:
with_reranker_metadata_filters_result_df = bounded_evaluation(with_reranker_metadata_filters_dataset)

#### 3.5 Preparar DataFrame de comparação

In [ ]:
import pandas as pd

# Create the side-by-side DataFrame
comparison_df = pd.DataFrame({
    'question': without_reranker_result_df['question'],
    'without_reranker_answer': without_reranker_result_df['answer'],
    'with_reranker_answer': with_reranker_result_df['answer'],
    'with_reranker_metadata_answer': with_reranker_metadata_filters_result_df['answer'],
    
    'without_reranker_answer_correctness': without_reranker_result_df['answer_correctness'],
    'with_reranker_answer_correctness': with_reranker_result_df['answer_correctness'],
    'with_reranker_metadata_correctness': with_reranker_metadata_filters_result_df['answer_correctness'],
    })

In [ ]:
pd.options.display.max_colwidth = 1000
comparison_df

In [ ]:
# Calculate average correctness
without_reranker_avg_correctness = without_reranker_result_df['answer_correctness'].mean()
with_reranker_avg_correctness = with_reranker_result_df['answer_correctness'].mean()
with_reranker_metadata_avg_correctness = with_reranker_metadata_filters_result_df['answer_correctness'].mean()

print(f"\nAverage Correctness without Reranker: {without_reranker_avg_correctness:.4f}")
print(f"Average Correctness with Reranker: {with_reranker_avg_correctness:.4f}")
print(f"Average Correctness with Reranker and metadata filter: {with_reranker_metadata_avg_correctness:.4f}")

### 2.7 Clean up
Por favor, certifique-se de descomentar e executar as células abaixo para excluir os recursos criados neste notebook.

In [ ]:
# delete local directory
import shutil

dir_path = "sec-10-k" # Replace with the actual path

try:
    shutil.rmtree(dir_path)
    print(f"Directory '{dir_path}' and its contents have been deleted successfully.")
except FileNotFoundError:
    print(f"Directory '{dir_path}' not found.")
except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")


In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")
